In [ ]:
#importing libraries

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import librosa
import matplotlib.pyplot as plt

import os
from PIL import Image
from pathlib import Path
import csv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
import keras
import librosa.display

# songname = f'/home/em/Desktop/genres/blues/blues.00000.wav'
# y, sr = librosa.load(songname, mono=True, duration=2, offset=0)
# ps = librosa.feature.melspectrogram(y=y, sr=sr, hop_length = 256, n_fft = 512, n_mels=128)
# ps = librosa.power_to_db(ps**2)
# ps.shape

In [ ]:
# Load dataset from the file
import pickle
with open('/content/drive/MyDrive/music classification/dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

print("Dataset loaded!")

Dataset loaded!


In [ ]:
dataset = []
genres = {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
        'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}

# /content/drive/MyDrive/music classification/genres_original

for genre, genre_number in genres.items():
    for filename in os.listdir(f'/content/drive/MyDrive/music classification/genres_original/{genre}'):
        songname = f'/content/drive/MyDrive/music classification/genres_original/{genre}/{filename}'
        for index in range(15):
            y, sr = librosa.load(songname, mono=True, duration=2, offset=index*2)
            ps = librosa.feature.melspectrogram(y=y, sr=sr, hop_length = 256, n_fft = 512, n_mels=64)
            ps = librosa.power_to_db(ps**2)
            dataset.append( (ps, genre_number) )

In [ ]:
import pickle

# Save dataset to a file
with open('/content/drive/MyDrive/music classification/dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)

print("Dataset saved!")


Dataset saved!


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install ffmpeg librosa soundfile

In [ ]:
print(len(dataset))

15000


In [ ]:
import random
import librosa
import os
import numpy as np
import keras

# Import the Input class from tensorflow.keras.layers
from tensorflow.keras.layers import Input

random.shuffle(dataset)

train = dataset[:11000]
valid = dataset[11000:13000]
test = dataset[13000:]

X_train, Y_train = zip(*train)
X_valid, Y_valid = zip(*valid)
X_test, Y_test = zip(*test)

# Reshape for CNN input
# Calculate the desired number of time frames (n_frames)
n_frames = 173  # You may need to adjust this value based on your data

# Pad or truncate melspectrograms to have the desired number of time frames
X_train = np.array([np.pad(x, ((0, 0), (0, n_frames - x.shape[1])), 'constant') if x.shape[1] < n_frames
                   else x[:, :n_frames] for x in X_train])
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1) # Add an extra dimension for channels

X_valid = np.array([np.pad(x, ((0, 0), (0, n_frames - x.shape[1])), 'constant') if x.shape[1] < n_frames
                   else x[:, :n_frames] for x in X_valid])
X_valid = X_valid.reshape(X_valid.shape[0], X_valid.shape[1], X_valid.shape[2], 1) # Add an extra dimension for channels


X_test = np.array([np.pad(x, ((0, 0), (0, n_frames - x.shape[1])), 'constant') if x.shape[1] < n_frames
                   else x[:, :n_frames] for x in X_test])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1) # Add an extra dimension for channels

# One-Hot encoding for classes
Y_train = np.array(keras.utils.to_categorical(Y_train, 10))
Y_valid = np.array(keras.utils.to_categorical(Y_valid, 10))
Y_test = np.array(keras.utils.to_categorical(Y_test, 10))

In [ ]:
len(X_train)
X_train.shape
n_features = X_train.shape[2]
input_shape = (None, X_train.shape[1])
print(input_shape)
model_input = Input(input_shape, name='input')
print(model_input)
X_train.shape

(None, 64)
<KerasTensor shape=(None, None, 64), dtype=float32, sparse=False, name=input>


(11000, 64, 173, 1)

In [ ]:
import keras
# Classification with Keras
# Building our Network
from keras import models
from keras import layers
from keras import Input, backend, Model
# LSTM is now directly imported from keras.layers
from keras.layers import LSTM
from keras.models import Sequential
from keras.layers import Dense, Lambda
from keras.layers import Activation
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dropout
from keras.layers import Flatten, GRU
from keras.layers import BatchNormalization
from keras.layers import AveragePooling2D
from keras.layers import TimeDistributed
from keras import ops
from tensorflow.keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau

# Define the ReduceLROnPlateau callback
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

from keras import regularizers
# model = Sequential()

# model.add(Conv2D(20, (5, 5), input_shape=(64, 173, 1),
#                  activation="relu", strides=1, padding="valid"))
# model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Conv2D(50, (5, 5), use_bias=50))
# model.add(MaxPooling2D(pool_size=(2, 2)))
# model.add(Flatten())
# model.add(Dense(20, activation="relu"))
# model.add(Lambda(lambda x: ops.expand_dims(x, axis=-1)))
# model.add(LSTM(512, activation="relu", return_sequences=False))
# model.add(Dense(10, activation = "softmax"))
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(64, 173, 1), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001)))
model.add(Dropout(0.5))
model.add(Dense(10, activation="softmax"))
model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 64, 173, 32)         │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 64, 173, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 32, 86, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 32, 86, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 32, 86, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 16, 43, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 44032)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       5,636,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,656,714 (21.58 MB)

 Trainable params: 5,656,522 (21.58 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
from keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate = 1e-5), loss="categorical_crossentropy", metrics=['accuracy'])

In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=20, verbose=2)
history = model.fit(X_train, Y_train, epochs=100, batch_size=64, validation_data= (X_test, Y_test), callbacks=[early_stopping,lr_scheduler])

Epoch 1/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.3892 - loss: 1.9797 - val_accuracy: 0.5030 - val_loss: 1.7087 - learning_rate: 1.0000e-05
Epoch 2/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.4643 - loss: 1.7854 - val_accuracy: 0.5300 - val_loss: 1.5893 - learning_rate: 1.0000e-05
Epoch 3/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.5125 - loss: 1.6600 - val_accuracy: 0.5595 - val_loss: 1.5191 - learning_rate: 1.0000e-05
Epoch 4/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.5396 - loss: 1.5679 - val_accuracy: 0.5850 - val_loss: 1.4704 - learning_rate: 1.0000e-05
Epoch 5/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.5720 - loss: 1.4839 - val_accuracy: 0.6015 - val_loss: 1.4307 - learning_rate: 1.0000e-05
Epoch 6/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.6130 - loss: 1.3823 - val_accuracy: 0.6235 - val_loss: 1.3922 - learning_rate: 1.0000e-05
Epoch 7/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/ste

In [ ]:
# Save the model to an H5 file
model.save('/content/drive/MyDrive/music classification/music_model_lr.h5')
print("Model saved as model.h5!")

Model saved as model.h5!


# **for loading model**

In [ ]:
# from keras.models import load_model

# # Load the model
# model = load_model('/content/drive/MyDrive/music_classification/model.h5')
# print("Model loaded successfully!")


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,8))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['accuracy','val_accuracy'])
plt.show()

plt.figure(figsize=(12,8))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['loss','val_loss'])
plt.show()
train_loss, train_acc = model.evaluate(X_train, Y_train, verbose = 2)
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose = 2)
print('Training accuracy:', train_acc)
print('Test accuracy:', test_acc)